##### (1) Response 대상 문장 추출 - ner_decision_results.review_candidate = true

In [2]:
import pandas as pd
from src.config import get_postgre_connect

query = """
    SELECT DISTINCT data_version, press, doc_id
    FROM public.ner_decision_results
    WHERE review_candidate IS TRUE
"""

with get_postgre_connect() as conn:
    response_targets = pd.read_sql_query(query, conn)

C:\Users\PC2512\AppData\Local\Temp\ipykernel_23760\1979975490.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  response_targets = pd.read_sql_query(query, conn)


In [3]:
response_targets

,data_version,press,doc_id
0,20261117_192958_강원일보,강원일보,NLRW2600000001.10006
1,20261117_192958_강원일보,강원일보,NLRW2600000001.10008
2,20261117_192958_강원일보,강원일보,NLRW2600000001.10014
3,20261117_192958_강원일보,강원일보,NLRW2600000001.1002
4,20261117_192958_강원일보,강원일보,NLRW2600000001.10024
...,...,...,...
136807,20261117_192337_충청일보,충청일보,NLRW2600000004.991
136808,20261117_192337_충청일보,충청일보,NLRW2600000004.9938
136809,20261117_192337_충청일보,충청일보,NLRW2600000004.9978
136810,20261117_192337_충청일보,충청일보,NLRW2600000004.9988


##### # (2) (1) 문장의 기사ID = 원본 데이터에서 전체 문장 추출 (기사 단위로 입력하기 위해) - ner_documents

In [12]:
CURRENT_PRESS = "스포츠투데이"  # 조회할 언론사

query = """
    WITH target_articles AS (
        SELECT DISTINCT data_version, press, doc_id
        FROM public.ner_decision_results
        WHERE review_candidate IS TRUE
          AND press = %s
    )
    SELECT
        t.data_version, t.press, 
        n.doc_id, n.paragraph_id, n.sentence_id, n.form,
        n.paragraph_order, n.sentence_order,
        n.document_title, n.author, n.publisher, n.date, n.topic, n.original_topic,
        d.review_candidate, d.detected_entities, d.candidate_rule_ids,
        rule_evidence.rule_evidence,
        d.tagging_error_probability, d.tagging_error_result, d.tagging_error_confidence,
        d.tagging_merge_probability, d.tagging_merge_result, d.tagging_merge_confidence
    FROM target_articles AS t
    JOIN public.ner_documents AS n
      ON n.doc_id = t.doc_id
    LEFT JOIN public.ner_decision_results AS d
      ON d.data_version = t.data_version
     AND d.press = t.press
     AND d.doc_id = n.doc_id
     AND d.sentence_id = n.sentence_id
    LEFT JOIN LATERAL (
        SELECT jsonb_agg(
            jsonb_build_object(
                'rule_id', item.rule_id,
                'pattern', item.rule_result ->> 'pattern'
            ) ORDER BY item.rule_id
        ) AS rule_evidence
        FROM public.ner_rule_results AS rr
        CROSS JOIN LATERAL jsonb_each(rr.rule_results)
            AS item(rule_id, rule_result)
        WHERE rr.data_version = t.data_version
          AND rr.doc_id = n.doc_id
          AND rr.sentence_id = n.sentence_id
          AND (item.rule_result ->> 'detected')::boolean IS TRUE
    ) AS rule_evidence ON TRUE
    ORDER BY t.press, n.doc_id, n.sentence_order, n.sentence_id
"""

with get_postgre_connect() as conn:
    response_documents = pd.read_sql_query(query, conn, params=(CURRENT_PRESS,))

len(response_documents.doc_id.unique())

C:\Users\PC2512\AppData\Local\Temp\ipykernel_23760\3660741480.py:46: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  response_documents = pd.read_sql_query(query, conn, params=(CURRENT_PRESS,))


2004

- response_documents 를 가지고 프롬프트 입력에 사용될 데이터 생성

In [13]:
import json
import re

ENTITY_PATTERN = re.compile(r"<([^<>:]+):\s*(PS|LC|OG|AF|DT|EV)>")

def clean_value(value):
    return None if pd.api.types.is_scalar(value) and pd.isna(value) else value

def split_rule_ids(value):
    value = clean_value(value)
    return [item.strip() for item in str(value).split(",") if item.strip()] if value else []

def normalize_rule_evidence(value):
    value = clean_value(value)
    if isinstance(value, str):
        return json.loads(value) if value else []
    return value if isinstance(value, list) else []

documents = []
for (data_version, press, doc_id), doc_frame in response_documents.groupby(
    ["data_version", "press", "doc_id"], sort=False
):
    paragraphs = []
    for (paragraph_id, paragraph_order), paragraph_frame in doc_frame.groupby(
        ["paragraph_id", "paragraph_order"], sort=False, dropna=False
    ):
        sentences = []
        for _, row in paragraph_frame.sort_values("sentence_order").iterrows():
            sentence_id = row["sentence_id"]
            form = clean_value(row["form"]) or ""
            is_target = bool(clean_value(row["review_candidate"]) or False)
            rule_ids = split_rule_ids(row["candidate_rule_ids"])
            rule_evidence = normalize_rule_evidence(row["rule_evidence"])
            entities = [
                {
                    "entity_id": f"{sentence_id}.E{i}",
                    "target_span": match.group(0),
                    "target_type": match.group(2),
                    "rule_evidence": [],
                }
                for i, match in enumerate(ENTITY_PATTERN.finditer(form), start=1)
            ] if is_target else []
            decision = None
            if is_target:
                decision = {
                    "review_candidate": True,
                    "detected_entities": clean_value(row["detected_entities"]),
                    "candidate_rule_ids": rule_ids,
                    "tagging_error_probability": clean_value(row["tagging_error_probability"]),
                    "tagging_error_result": clean_value(row["tagging_error_result"]),
                    "tagging_error_confidence": clean_value(row["tagging_error_confidence"]),
                    "tagging_merge_probability": clean_value(row["tagging_merge_probability"]),
                    "tagging_merge_result": clean_value(row["tagging_merge_result"]),
                    "tagging_merge_confidence": clean_value(row["tagging_merge_confidence"]),
                }
            sentences.append({
                "id": sentence_id,
                "form": form,
                "entities": entities,
                "sentence_rule_evidence": rule_evidence,
                "decision": decision,
            })
        paragraphs.append({"id": paragraph_id, "sentence": sentences})

    first = doc_frame.iloc[0]
    documents.append({
        "id": doc_id,
        "data_version": data_version,
        "metadata": {
            "title": clean_value(first["document_title"]),
            "author": clean_value(first["author"]),
            "publisher": press,
            "date": clean_value(first["date"]),
            "topic": clean_value(first["topic"]),
            "original_topic": clean_value(first["original_topic"]),
        },
        "paragraph": paragraphs,
    })

response_documents = {"document": documents}

##### (3) Response 배치 실행 테스트

(2)에서 구성한 `response_documents`를 테스트와 언론사별 실행에 공통으로 사용한다. 아래에는 이전 실행 출력이 남아 있다. 새 실행 라벨은 `run_response_2_t3_test1`이며, 기본 설정에서는 강원일보 저장 응답 2건만 로컬 재현한다.

In [14]:
import json
import os
from collections import Counter
from importlib import reload
from itertools import islice
from urllib.parse import quote

import pyarrow.parquet as pq

from src.ner_review import response_batch, response_state, result as review_result
from src.ner_review.response import BASE_DIR
from src.ner_review.storage import REVIEW_COLUMNS

PROVIDER = os.getenv("API_PROVIDER", "OPENAI").upper()  # "OPENAI" 또는 "OPENROUTER"
MODEL = os.environ["MODEL_NAME"]  # OpenRouter 예: "openai/gpt-5.1"
MODE = os.getenv("RESPONSE_MODE", "batch")  # batch / parallel / sync(단건 순차)
TEST_LIMIT = 10
BATCH_SIZE = 5
MAX_WORKERS = 4
MAX_RETRIES = 3
TIMEOUT = 120
BACKOFF = 2
MAX_BACKOFF = 60
POLL_SECONDS = 30

#---------------------------- 실행 전 체크! -------------------------------
RUN_TEST = False # True로 설정하면 테스트 배치 호출
FORCE = False    # 선택 문서 전체 재처리. 중단 후 이어갈 때는 False
RUN_LABEL = "run_response_2_t3" 
# 새 실행 라벨. 중단 후 재개할 때는 같은 라벨 사용
# 최근 실행명: "run_response_2_t3"
# 최근 테스트 실행명: "run_response_2_t2_test2"  
#------------------------------------------------------------------------

documents_by_unit = {}
for doc in response_documents["document"]:
    unit = (doc["data_version"], doc["metadata"]["publisher"])
    documents_by_unit.setdefault(unit, {})[doc["id"]] = doc
units = pd.DataFrame(sorted(documents_by_unit), columns=["data_version", "press"])
display(units.reset_index(drop=True))

,data_version,press
0,20261117_224604_스포츠투데이,스포츠투데이


In [15]:
def article_ids(version, press):
    ids = sorted(documents_by_unit.get((version, press), {}))
    if not ids:
        raise ValueError(f"선택한 언론사와 버전에 대상 기사가 없습니다: {press}, {version}")
    return ids


def prepared_documents(ids, *, data_version, press):
    by_id = documents_by_unit[data_version, press]
    return [by_id[key] for key in ids]


def output_root(version):
    run_name = f"{RUN_LABEL}_{version}_{PROVIDER}_{MODEL}"
    # 한글은 그대로 두고 모델명의 슬래시 등 경로 특수문자만 인코딩한다.
    run_id = "".join(quote(char, safe="") if char.isascii() else char for char in run_name)
    return BASE_DIR / "outputs" / str(version) / run_id


def verify_response_processor():
    # 기존 커널에서도 배치가 참조하는 후처리와 실행 기록 모듈을 함께 갱신한다.
    reload(review_result)
    reload(response_state)
    reload(response_batch)
    archive = BASE_DIR / "src/ner_review/fixtures/gangwon_t2_expansions.json"
    saved = json.loads(archive.read_text(encoding="utf-8"))
    for case in saved["cases"]:
        sentence = case["sentence"]
        row, = review_result.build_result(case["response"], [sentence])["reviews"]
        if (row["entity_id"] != sentence["entities"][0]["entity_id"]
                or row["corrected_span"] != case["expected_span"] or row["review_needed"]):
            raise RuntimeError(f"저장 응답 재현 실패: {row['entity_id']}, {row['corrected_span']}")
    print("강원일보 저장 응답 2건 재현 완료 · 후처리 SHA-256", review_result.POSTPROCESS_SHA256)


def execute(ids, version, press, batch_size, *, force=False):
    verify_response_processor()
    return response_batch.run_response_batch(
        ids, output_root(version), data_version=version, press=press,
        load_documents=prepared_documents,
        provider=PROVIDER, model=MODEL, mode=MODE, batch_size=batch_size,
        max_workers=MAX_WORKERS, max_retries=MAX_RETRIES, timeout=TIMEOUT,
        backoff=BACKOFF, max_backoff=MAX_BACKOFF, poll_seconds=POLL_SECONDS, force=force,
    )


if units.empty:
    raise ValueError("(2)의 조회 결과에 review_candidate 대상 기사가 없습니다")
TEST_VERSION, TEST_PRESS = units.iloc[0]  # 위 목록에서 직접 지정 가능
if TEST_LIMIT < 1:
    raise ValueError("TEST_LIMIT은 1 이상이어야 합니다")
test_ids = article_ids(TEST_VERSION, TEST_PRESS)[:TEST_LIMIT]
test_dir = output_root(TEST_VERSION) / "reports" / TEST_PRESS
print(TEST_PRESS, TEST_VERSION, f"테스트 대상 {len(test_ids)}건", PROVIDER, MODEL, MODE, test_dir)

스포츠투데이 20261117_224604_스포츠투데이 테스트 대상 10건 OPENAI gpt-5.1 batch C:\work\ner\outputs\20261117_224604_스포츠투데이\run_response_2_t3_20261117_224604_스포츠투데이_OPENAI_gpt-5.1\reports\스포츠투데이


OpenAI의 `batch`는 서버 완료까지 polling한다. OpenRouter의 `batch`와 `parallel`은 로컬 worker를 사용한다. 두 provider 모두 `sync`를 선택하면 한 번에 한 문서를 호출한다. 같은 provider·모델·버전의 테스트 성공은 (4)에서도 건너뛴다.

In [9]:
if RUN_TEST:
    test_summary = execute(test_ids, TEST_VERSION, TEST_PRESS, BATCH_SIZE, force=FORCE)
    display(pd.Series(test_summary))
    print("이번 실행에서 건너뛴 기사:", test_summary["skipped"])
else:
    verify_response_processor()
    print("로컬 검증 완료. RUN_TEST=True로 설정하면 선택한 기사만 호출합니다.")

강원일보 저장 응답 2건 재현 완료 · 후처리 SHA-256 589a716960096f4e744d5580e2c19d80fd1a25085b24f8d68485e260c29362b3


run_id                run_response_2_t2_test2_20261117_192958_강원일보_O...
provider                                                         OPENAI
model                                                           gpt-5.1
data_version                                       20261117_192958_강원일보
prompt_version                                            ner-review-v3
prompt_sha256         27b9ee9710988b988bd37861065c9ca148bfe79f9cc1f7...
postprocess_sha256    589a716960096f4e744d5580e2c19d80fd1a25085b24f8...
load_dt                                                      2026-11-25
press                                                              강원일보
mode                                                              batch
requested_mode                                                    batch
started_at                             2026-11-25T04:23:29.912887+00:00
selected                                                             10
skipped                                                         

이번 실행에서 건너뛴 기사: 0


검증 셀은 로컬 파일을 읽는다. 같은 테스트 셀을 `FORCE=False`로 다시 실행하면 성공 기사는 `skipped`로 집계되고 실패·미처리 기사만 호출된다. `summary.json`의 상태별 건수는 해당 출력 폴더의 누적 문서 수다.

In [9]:
def verify_results(directory):
    if not (directory / "summary.json").exists():
        print("저장된 실행 결과가 없습니다.", directory)
        return
    summary = json.loads((directory / "summary.json").read_text(encoding="utf-8"))
    display(pd.Series(summary))
    assert summary["total"] == sum(summary[k] for k in ("completed", "failed", "pending"))
    with (directory / "results.jsonl").open(encoding="utf-8") as src:
        completed = Counter(json.loads(line)["doc_id"] for line in src)
    assert len(completed) == summary["completed"] and all(n == 1 for n in completed.values())
    row_count = 0
    finished_docs = set()
    previous_doc = None
    with (directory / "reviews.jsonl").open(encoding="utf-8") as src:
        for line in src:
            row = json.loads(line)
            doc = row["doc_id"]
            if doc != previous_doc:
                assert doc not in finished_docs, f"문서 결과 중복: {doc}"
                finished_docs.add(doc)
                previous_doc = doc
            assert (doc in completed) == (row["llm_review_status"] == "reviewed")
            row_count += 1
    assert completed.keys() <= finished_docs
    if summary["status"].startswith("completed"):
        parquet = pq.ParquetFile(directory / "ner_reviews.parquet")
        assert parquet.schema_arrow.names == REVIEW_COLUMNS
        assert parquet.metadata.num_rows == row_count
    with (directory / "failures.jsonl").open(encoding="utf-8") as src:
        display(pd.DataFrame(json.loads(line) for line in islice(src, 5)))
    print(f"성공 문서 {len(completed)}건, 검수 결과 {row_count}행", directory)

# verify_results(test_dir)

##### (4) 언론사별 배치 진행

(2)에서 선택한 언론사를 실행한다. 언론사를 바꾸려면 (2)의 `CURRENT_PRESS`를 변경하고 조회·변환 셀과 (3)의 설정 셀을 다시 실행한다. 실패 후 재개할 때는 같은 `RUN_LABEL`과 `FORCE=False`를 사용한다. 호출 제한이 반복되면 `MAX_WORKERS`와 배치 크기를 낮춰 재실행할 수 있다.

In [ ]:
TEST_VERSION, TEST_PRESS = units.iloc[0]

DATA_VERSION = TEST_VERSION
PRESS_BATCH_SIZE = 500 # 테스트시 20

press_ids = article_ids(DATA_VERSION, CURRENT_PRESS)
press_dir = output_root(DATA_VERSION) / "reports" / CURRENT_PRESS
print(CURRENT_PRESS, DATA_VERSION, f"전체 기사 {len(press_ids)}건", press_dir)

press_summary = execute(press_ids, DATA_VERSION, CURRENT_PRESS, PRESS_BATCH_SIZE, force=FORCE)
display(pd.Series(press_summary))
verify_results(press_dir)